# 01 - Exploring the screw dataset (EDA)

Goal: look at the data before building any model.

1. Check the counts (train/val split, test images per defect type).
2. Look at good screws: random angles and positions.
3. Look at every defect type together with its mask.
4. Measure how big the defects are, and how many defect pixels are left after resizing.
5. Compare the brightness of good and defective images.

Every figure is saved to `outputs/figures/`. Nothing is trained here.
Looking at the defective test images is allowed here because we only *look* at them;
we never use them to train or to choose a threshold.

In [ ]:
# This notebook runs from notebooks/, so we add the project root to sys.path
# first. After that `from src...` works like in every other file.
import sys
from pathlib import Path

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import matplotlib.pyplot as plt

from src.paths import FIGURES
from src.data import (DEFECT_TYPES, get_split, list_test,
                      load_image, load_mask, load_images)

print("project root:", ROOT)
print("figures go to:", FIGURES)

## 1. Counts

In [ ]:
split = get_split()
test = list_test()

print("train good:", len(split["train"]), "| val good:", len(split["val"]))
counts = {}
for t in test:
    counts[t["defect"]] = counts.get(t["defect"], 0) + 1
print("test images:", len(test))
for name, n in counts.items():
    print(f"  {name:18s} {n}")

## 2. Good screws

The screws lie at random angles and positions, so a model cannot just memorise
"the screw is always here". Full resolution (1024 px) is shown here.

In [ ]:
rng = np.random.default_rng(0)
picks = rng.choice(split["train"], size=8, replace=False)

fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for ax, p in zip(axes.flat, picks):
    ax.imshow(load_image(p, size=1024), cmap="gray", vmin=0, vmax=1)
    ax.set_title(p.split("/")[-1], fontsize=9)
    ax.axis("off")
fig.suptitle("Good screws (train): random angle and position")
fig.tight_layout()
fig.savefig(FIGURES / "eda_01_good_samples.png", dpi=100)
plt.show()

## 3. Every defect type with its mask

Left: the image. Right: the same image with the true defect region (mask) in red.
Notice how small most defects are.

In [ ]:
by_type = {d: [t for t in test if t["defect"] == d] for d in DEFECT_TYPES}

fig, axes = plt.subplots(len(DEFECT_TYPES), 4, figsize=(12, 3 * len(DEFECT_TYPES)))
for row, d in zip(axes, DEFECT_TYPES):
    for k in range(2):                      # two examples per type
        t = by_type[d][k]
        img = load_image(t["file"], size=1024)
        mask = load_mask(t["mask"], size=1024)
        row[2 * k].imshow(img, cmap="gray", vmin=0, vmax=1)
        row[2 * k].set_title(f"{d} / {t['file'].split('/')[-1]}", fontsize=8)
        row[2 * k + 1].imshow(img, cmap="gray", vmin=0, vmax=1)
        row[2 * k + 1].imshow(np.ma.masked_where(mask == 0, mask), cmap="autumn", alpha=0.6)
        row[2 * k + 1].set_title(f"mask: {100 * mask.mean():.2f}% of image", fontsize=8)
for ax in axes.flat:
    ax.axis("off")
fig.tight_layout()
fig.savefig(FIGURES / "eda_02_defect_examples.png", dpi=80)
plt.show()

## 4. How big are the defects?

Defect area = share of the image covered by the mask, measured at full resolution.

In [ ]:
defects = [t for t in test if t["label"] == 1]
area = {d: [] for d in DEFECT_TYPES}
for t in defects:
    area[t["defect"]].append(100 * load_mask(t["mask"], size=1024).mean())

print(f"{'defect':18s} {'n':>3s} {'median %':>9s} {'min %':>7s} {'max %':>7s}")
for d in DEFECT_TYPES:
    a = np.array(area[d])
    print(f"{d:18s} {len(a):3d} {np.median(a):9.2f} {a.min():7.2f} {a.max():7.2f}")
all_area = np.concatenate([area[d] for d in DEFECT_TYPES])
print(f"all defects: median {np.median(all_area):.2f}%, "
      f"{(all_area < 1).mean() * 100:.0f}% of defects cover less than 1% of the image")

fig, ax = plt.subplots(figsize=(9, 4))
ax.boxplot([area[d] for d in DEFECT_TYPES], showfliers=True)
ax.set_xticks(range(1, len(DEFECT_TYPES) + 1), DEFECT_TYPES, rotation=15)
ax.set_ylabel("defect area (% of image)")
ax.set_title("Defect size per type")
ax.grid(axis="y", alpha=0.3)
fig.tight_layout()
fig.savefig(FIGURES / "eda_03_defect_area.png", dpi=100)
plt.show()

### What resizing does to tiny defects

The models work on smaller images (256 px by default). Here we count how many
defect pixels are left at each size. If a defect is only a few pixels at 256 px,
it may be too small to detect, and that is the reason to also try a bigger size.

In [ ]:
sizes = [128, 256, 512, 1024]
pixels = {s: np.array([load_mask(t["mask"], size=s).sum() for t in defects]) for s in sizes}

print(f"{'size':>5s} {'median px':>10s} {'smallest':>9s} {'defects < 20 px':>16s}")
for s in sizes:
    p = pixels[s]
    print(f"{s:5d} {np.median(p):10.0f} {p.min():9d} {(p < 20).sum():16d}")

fig, ax = plt.subplots(figsize=(7, 4))
ax.boxplot([pixels[s] for s in sizes])
ax.set_xticks(range(1, len(sizes) + 1), [f"{s} px" for s in sizes])
ax.set_yscale("log")
ax.set_ylabel("defect pixels (log scale)")
ax.set_title("Defect pixels left after resizing")
ax.grid(axis="y", alpha=0.3)
fig.tight_layout()
fig.savefig(FIGURES / "eda_04_defect_pixels_vs_size.png", dpi=100)
plt.show()

## 5. Brightness

Mean brightness per image, for the three groups of images. If one group is
clearly brighter or darker, a model may react to the brightness instead of the defect.
Also look at the background in the good samples above: small dust specks can
cause false alarms.

In [ ]:
groups = {
    "train good": split["train"],
    "test good": [t["file"] for t in test if t["label"] == 0],
    "test defective": [t["file"] for t in defects],
}
mean_brightness = {g: load_images(files).mean(axis=(1, 2)) for g, files in groups.items()}
bins = np.linspace(min(m.min() for m in mean_brightness.values()),
                   max(m.max() for m in mean_brightness.values()), 40)

fig, ax = plt.subplots(figsize=(8, 4))
for g, m in mean_brightness.items():
    ax.hist(m, bins=bins, alpha=0.5, density=True, label=f"{g} (n={len(m)})")
    print(f"{g:15s} mean brightness {m.mean():.3f} +/- {m.std():.3f}")
ax.set_xlabel("mean pixel value (0 = black, 1 = white)")
ax.set_ylabel("density")
ax.set_title("Mean brightness per image")
ax.legend()
fig.tight_layout()
fig.savefig(FIGURES / "eda_05_brightness.png", dpi=100)
plt.show()

## Summary

- 320 good training images (280 train / 40 val), 160 test images (41 good, 119 defective).
- Screws lie at random angles and positions.
- Defects are tiny: look at the median area above, and at how few pixels remain at 256 px.
  This is why we should also try a larger input size.
- Brightness: the **good test images are darker** than both the training images and the
  defective test images (see the numbers in section 5). A model trained on the brighter
  training images may therefore give the good test screws higher anomaly scores (false alarms).
  Keep this in mind when reading the results, and check it in the robustness step.

Figures saved: `eda_01` to `eda_05` in `outputs/figures/`.